In [ ]:
import numpy as np 
from pathlib import Path
import csv
import os
import glob
import keras
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers,models,regularizers,constraints
import seaborn as sns
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix,classification_report
from tensorflow.keras.callbacks import ReduceLROnPlateau,EarlyStopping
import time
from scipy.signal import fftconvolve
from sklearn.model_selection import train_test_split
import re
plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus']=False
import math
import shutil
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold


In [ ]:

def robust_read_rf_csv(file_path,length = 500000):
    samples = []
    total_exception_count = 0
    total_point_count = 0
    
    with open(file_path,'r') as f:
        for idx,line in enumerate(f):
            fields = line.strip().split(',')
            values = []
            for item in fields:
                try:
                    value = float(item)
                    values.append(value)
                except Exception:
                    values.append(0)
            if len(values)>length:
                values = values[:length]
            elif len(values)<length:
                values = values + [0]*(length-len(values))

            values = np.array(values,dtype = np.float32)

            values = values/(2**15)

            mask = np.isnan(values)|np.isinf(values)|(np.abs(values)>1)
            exception_count = np.sum(mask)
            total_exception_count += exception_count
            total_point_count += len(values)
            values[mask]=0.0

            v_max = np.max(values)
            v_min = np.min(values)
            v_mean = np.mean(values)
            exception_ratio = exception_count/len(mask)

            samples.append(values)
    arr = np.array(samples,dtype=np.float32)

    total_exception_ratio = total_exception_count/total_point_count
    arr_max = np.max(arr)
    arr_min = np.min(arr)
    arr_mean = np.mean(arr)
    print(f'[robust_read_rf_csv] {file_path}: max = {arr_max:.2f} min = {arr_min:.2f} mean = {arr_mean:.2f}')
    print(f'[robust_read_rf_csv] {file_path}: Total number of outliers={total_exception_count} Ratio={total_exception_ratio:.6f}')
    return arr
    

In [ ]:
t0 = time.time()
num_classes = 4 # Class
data_folder = 'Experiment_dataset' # 
drone_types = ['Air3S','Mavic2pro','Mini3pro','Mini5pro'] 
RF_suffixes = ('RF_I.csv','RF_Q.csv')
seq_length = 500000
height_folders = ['1H','2H']


def load_microphone_signal(
    data_folder,
    drone_types,
    RF_suffixes,
    num_classes,
    height_folders = ['1H','2H']
    ):

    print("Reading Data")
    all_labels = [] 
    all_RF     = []

    for idx,drone_type in enumerate(drone_types):
        label = np.zeros(num_classes)
        label[idx] = 1
        type_folder = os.path.join(data_folder,drone_type)
        if not os.path.exists(type_folder):
            print(f"{type_folder} not exist,skip")
            continue
        for height in height_folders:
            height_path = os.path.join(type_folder,height)
            if not os.path.exists(height_path):
                print(f'{height_path} not exist,skip')
                continue
            # Reading passed_index(Removed samples are contaminated by burst noise)
            passed_index_file = os.path.join(height_path,'passed_indices.csv')
            passed_indices = pd.read_csv(passed_index_file)["passed_index"].to_numpy()
            passed_indices = passed_indices-1
            # Reading RF Data
            rf_I_file = os.path.join(height_path,"RF_I.csv")
            rf_Q_file = os.path.join(height_path,"RF_Q.csv")
            if not (os.path.exists(rf_I_file) and os.path.exists(rf_Q_file)):
                print(f"{rf_I_file},{rf_Q_file} not exist,skip")
                continue
            try:
                rf_I_data = robust_read_rf_csv(rf_I_file,length=500_000)
                rf_Q_data = robust_read_rf_csv(rf_Q_file,length=500_000)
                if rf_I_data.shape != rf_Q_data.shape:
                    print(f"{height_path}: The number of RF_I and RF_Q samples is inconsistent,skip")
                    continue
                # Filter according to the index
                rf_I_data = rf_I_data[passed_indices]
                rf_Q_data = rf_Q_data[passed_indices]
                
                rf_data = np.stack([rf_I_data,rf_Q_data],axis=1)
                all_RF.append(rf_data)
                all_labels.extend([label]*rf_data.shape[0])
                print(f"{drone_type}-{height} remain RF samples:{rf_data.shape[0]}")
                
            except Exception as e:
                print(f'{height_path} RF_I/Q loading failed Error message: {e}')
                continue
                
    all_labels = np.asarray(all_labels)
    all_RF = np.vstack(all_RF)
    X_RF    = all_RF
    y_classify = all_labels
    print(f"Data reading completed! RF dataset shape:{X_RF.shape}, Category label shape: {y_classify.shape}")
    return X_RF,y_classify

def post_process_data(
    X_RF,
    y_classify,
    test_size = 0.2
):
    #Set a fixed seed for random shuffling
    X_RF = np.transpose(X_RF,(0,2,1))
    print(f"Data after transpose-RF data shape-{X_RF.shape}")
    np.random.seed(42)
    indices = np.random.permutation(X_RF.shape[0])
    print(indices.shape)
    X_RF = X_RF[indices]
    y_classify = y_classify[indices]

    # Divide test set and training set
    X_train_val_RF,X_test_RF,y_train_val_classify,y_test_classify = train_test_split(X_RF,y_classify,test_size=test_size,shuffle=True,random_state=random_state,stratify=y_classify)
    print(f"\n====Dataset Partitioning====")
    print(f"train+val:{X_train_val_RF.shape}--{y_train_val_classify.shape}")
    print(f"test:{X_test_RF.shape}--{y_test_classify.shape}")
    
    return X_train_val_RF,X_test_RF,y_train_val_classify,y_test_classify

X_RF,y_classify = load_microphone_signal(data_folder,drone_types,RF_suffixes,num_classes)
test_size = 0.2
random_state=42
X_train_val_RF,X_test_RF,y_train_val_classify,y_test_classify = post_process_data(X_RF,y_classify,test_size=test_size)
t1=time.time()
elapsed_time = t1-t0
print(f"Creating the dataset takes time: {elapsed_time:.4f} s")   

In [ ]:
# ==============  Establish the network ==================
class RF_Model(tf.keras.Model):
    def __init__(self,num_classes,physics_orth,input_shape_RF = (2,seq_length)):
        super().__init__()
        self.input_shape_RF = input_shape_RF
        self.I_channel = lambda x: x[:,:,0:1]
        self.Q_channel = lambda x: x[:,:,1:2]

        self.RF_conv = layers.Conv1D(16,8,strides=8,padding='same',activation='relu')
        self.RF_pool = layers.MaxPooling1D(pool_size=8)
        self.RF_global_pool = layers.GlobalAveragePooling1D()
        self.RF_concat = layers.Concatenate()
        self.RF_dense1 = layers.Dense(16,activation = 'relu')
        self.RF_dense2 = layers.Dense(8,activation = 'relu')
        self.physics_ortho_weight = tf.Variable(physics_orth,trainable=False,dtype=tf.float32,name='physics_orth_weight')
        self.class_output = layers.Dense(num_classes,activation='softmax',name='final_out_cls')
        self.loss_orth_metrics = tf.keras.metrics.Mean(name='loss_orth')
        
    def call(self,inputs,training=None):
        inputs_RF = inputs[0] if isinstance(inputs,tuple) else inputs # RF input
        
        # ====== Feature Extraction =========
        I_RF = self.I_channel(inputs_RF)
        Q_RF = self.Q_channel(inputs_RF)
        
        conv_I = self.RF_conv(I_RF)
        conv_Q = self.RF_conv(Q_RF)
        pooled_I = self.RF_pool(conv_I)
        pooled_Q = self.RF_pool(conv_Q)
        global_I = self.RF_global_pool(pooled_I)
        global_Q = self.RF_global_pool(pooled_Q)
        merged_RF = self.RF_concat([global_I,global_Q])
        x_RF = self.RF_dense1(merged_RF)
        RF_feat_32 = self.RF_dense2(x_RF)

        # Orth loss compute
        batch_size = tf.shape(conv_I)[0]
        I_flat = tf.reshape(conv_I,[batch_size,-1])
        Q_flat = tf.reshape(conv_Q,[batch_size,-1])
        I_norm = tf.nn.l2_normalize(I_flat,axis=1)
        Q_norm = tf.nn.l2_normalize(Q_flat,axis=1)
        cos_sim = tf.reduce_mean(tf.maximum(0.0,tf.reduce_sum(I_norm*Q_norm,axis=1)))
        total_RF_loss =self.physics_ortho_weight * cos_sim

        out_cls = self.class_output(RF_feat_32)
        self.loss_orth_metrics.update_state(cos_sim)
        self.add_loss(total_RF_loss)
        return out_cls


In [ ]:
def save_history_to_csv(history_data, filename,save_dir):
    with open(os.path.join(save_dir, filename), "w", newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(np.mat(history_data).transpose().tolist())
        
class LossHistory(tf.keras.callbacks.Callback):
    def on_train_begin(self,logs=None):
        self.orth_losses = []
    def on_epoch_end(self,epoch,logs=None):
        RF_layer = self.model
        self.orth_losses.append(RF_layer.loss_orth_metrics.result().numpy())
        if epoch % 200 == 0:
            print(f"Epoch {epoch},loss={logs.get('loss'):.4f},val_loss = {logs.get('val_loss'):.4f}")
            print("Learning rate:",self.model.optimizer.learning_rate.numpy())
            print(f"Orth_loss = {self.orth_losses[-1]}")

def save_result_fold(save_dir,history,history_loss,predictions,y_val,main_model):
    # === save data =============
    train_loss = history.history['loss']
    valid_loss = history.history['val_loss']
    plt.figure(figsize=(10, 6))
    plt.plot(train_loss, 'b', label='Training loss')
    plt.plot(valid_loss, 'r', label='Validation loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'Loss.png'))
    plt.close()
    
    Accuracy = history.history['accuracy']
    Val_Accuracy = history.history['val_accuracy']
    plt.figure(figsize=(10, 6))
    plt.plot(Accuracy, 'b', label='Training Accuracy')
    plt.plot(Val_Accuracy, 'r', label='Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'Accuracy.png'))
    plt.close()
    
    plt.figure(figsize=(10, 6))
    plt.plot(history_loss.orth_losses, 'b', label='Orth loss')
    plt.xlabel('Epochs')
    plt.ylabel('Orth Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'Orth Loss.png'))
    plt.close()

    
    save_history_to_csv(history.history['loss'], 'train_loss.csv',save_dir)
    save_history_to_csv(history.history['val_loss'], 'val_loss.csv',save_dir)
    save_history_to_csv(history.history['accuracy'], 'Class_Accuracy.csv',save_dir)
    save_history_to_csv(history.history['val_accuracy'], 'Val_Class_Accuracy.csv',save_dir)
    save_history_to_csv(history_loss.orth_losses, 'Orth_loss.csv',save_dir)


    y_pred_classes = np.argmax(predictions,axis=1)
    y_true_classes = np.argmax(y_val,axis=1)


    conf_matrix = confusion_matrix(y_true_classes, y_pred_classes)
    drone_types = ['Air3S','Mavic2pro','Mini3pro','Mini5pro']
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=drone_types, yticklabels=drone_types)
    plt.xlabel('Predict Label')
    plt.ylabel('Real Label')
    plt.title('Confusion Matrix')
    plt.savefig(os.path.join(save_dir, 'Confusion_Matrix.png'))
    plt.close()
    
    classification_rep = classification_report(y_true_classes, y_pred_classes, target_names=drone_types,digits=4)
    print("classification report:\n",classification_rep)
    with open(os.path.join(save_dir, 'Classification_Report.txt'),"w") as f:
        f.write(classification_rep)
        
    main_model.save(save_dir + '/my_main_model',save_format = 'tf')
    filename = 'main_model_summary.txt'
    file_path = os.path.join(save_dir,filename)
    with open(file_path,'w') as f:
        main_model.summary(print_fn = lambda x: f.write(x+'\n'))

In [ ]:
def get_model(seq_length, physics_orth, num_classes=4):
    tf.keras.backend.clear_session()
    model = RF_Model(num_classes=num_classes, physics_orth=physics_orth, input_shape_RF=(2, seq_length))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model
def train_with_history(model, X_train, y_train, X_val, y_val, epochs, batch_size, verbose=1):
    history_loss = LossHistory()
    result = model.fit(
        X_train, y_train, validation_data=(X_val, y_val),
        epochs=epochs, batch_size=batch_size,
        callbacks=[history_loss], verbose=verbose
    )
    return result, history_loss



In [ ]:
physics_orth = 0.0
nb_epoch = 1000
batch_size = 64
n_splits = 10

t0 = time.time()
loca = str(time.strftime('%Y-%m-%d-%H-%M-%S'))
base_path = f"Experiment_data_RF_close_classify/Batchsize_{batch_size}_Nbepoch_{nb_epoch}/{physics_orth}-{loca}/"
os.makedirs(base_path,exist_ok = True)

y_trainval_cls = np.argmax(y_train_val_classify,axis=1)
print('y_trainval_cls former 20 items:',y_trainval_cls[:20])
print('Category distribution:',np.bincount(y_trainval_cls))

skf = StratifiedKFold(n_splits = n_splits,shuffle=True,random_state=random_state)
acc_list = []
train_mi_list,val_mi_list = [],[]
print(f"\n=== {n_splits} fold cross-validation ====")

for fold,(train_idx,val_idx) in enumerate(skf.split(X_train_val_RF,y_trainval_cls),1):
    #========== 1. Initialize create file =========
    print(f"\n==== Fold {fold}/{n_splits} ====")
    fold_loca = str(time.strftime('%Y-%m-%d-%H-%M-%S'))
    fold_path = f"{fold_loca}-fold{fold}"
    fold_dir = os.path.join(base_path,fold_path)
    os.makedirs(fold_dir,exist_ok = True)

    # Split train/validation data for this fold from trainval set
    X_train,X_val = X_train_val_RF[train_idx],X_train_val_RF[val_idx]
    y_train,y_val = y_train_val_classify[train_idx],y_train_val_classify[val_idx]
    print(f"Train dataset size:{X_train.shape[0]}, validation dataset size:{X_val.shape[0]}")
    # ==========2. Network Training  ===============
    main_model = get_model(seq_length,physics_orth,num_classes)
    
    history,history_loss = train_with_history(main_model,X_train,y_train,X_val,y_val,nb_epoch,batch_size,verbose=0)

    val_acc = main_model.evaluate(X_val,y_val,batch_size=batch_size,verbose=0)[1]
    acc_list.append(val_acc)

    print(f"Fold {fold} Validation Accuracy: {val_acc:.4f}")
    predictions = main_model.predict(X_val,verbose=0)
    
    # ====== 3. save model result ================
    save_result_fold(fold_dir,history,history_loss,predictions,y_val,main_model)

    # ------------ save result --------------
    msg = (f"Fold {fold} Acc: {val_acc:.4f}\n")
    with open(os.path.join(fold_dir, "fold_result.txt"), "w", encoding="utf-8") as f:
        f.write(msg)

      
t1=time.time()
elapsed_time = t1-t0
print(f"Time consuming: {elapsed_time:.4f} s")

In [ ]:
# Summary
print("\n==== ALL-fold Results ====")
acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)
print(f"Acc: {acc_mean:.4f} ± {acc_std:.4f}")

summary_msg = (
    "==== ALL-fold Results ====\n"
    f"Acc: {acc_mean:.4f} ± {acc_std:.4f}\n"
)
summary_path = os.path.join(base_path, "results_summary.txt")


with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_msg)

print(f"Save to: {summary_path}")


In [ ]:
# ======================= 2. Retrain the final model and evaluate on the test set =====================
print("\nRetrain the model with the full train + validation set")

final_model = get_model(seq_length, physics_orth, num_classes)
final_history, final_history_loss = train_with_history(final_model, X_train_val_RF, y_train_val_classify, X_train_val_RF, y_train_val_classify,
                                                      nb_epoch, batch_size, verbose=0)
test_acc = final_model.evaluate(X_test_RF, y_test_classify, batch_size=batch_size, verbose=0)[1]
print(f"Test set accuracy: {test_acc:.4f}")
test_dir = os.path.join(base_path, "test")
os.makedirs(test_dir, exist_ok=True)
save_result_fold(test_dir, final_history, final_history_loss, final_model.predict(X_test_RF), y_test_classify, final_model)


# Save result
summary_msg_test = (
    "==== Test Results After ALL Fold ====\n"
    f"Test set Acc: {test_acc:.4f}\n"
)
summary_path = os.path.join(test_dir, "test_set_results_summary.txt")

with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_msg_test)

print(f"Save to : {summary_path}")

